# 8장 경사 하강법 — 미분, 학습률, 넘파이 구현, 배치·미니배치·확률적
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 작은 $h$로 기울기 재기, 편미분 함수 만들기
2. 업데이트 규칙으로 한 걸음씩 내려가고 학습률 바꿔 보기
3. 택시 기록 50건에서 $w$와 $b$를 함께 찾고, 표준화의 효과 확인하기
4. 배치, 미니배치, 확률적 경사 하강법 비교하고 `SGDRegressor` 써 보기

위에서부터 차례로 실행하세요.

## 1. 기울기 재기: 미분과 편미분
7장의 기록 세 건(예시)에서 절편을 $b = 4$로 두고, $w = 0.75$에서 $w$를 작은 $h$만큼 움직여 기울기를 잽니다. $h$를 줄일수록 미분 $-18.67$에 다가갑니다.

In [ ]:
import numpy as np

x = np.array([2.0, 4.0, 6.0])     # 거리(km), 7장의 기록 세 건
y = np.array([7.0, 8.0, 12.0])    # 요금(천 원)
def loss(w):                      # 절편은 b = 4로 둔다
    return np.mean((y - (w * x + 4)) ** 2)

w = 0.75
for h in [0.1, 0.01, 0.001, 0.0001]:
    slope = (loss(w + h) - loss(w)) / h
    print(f"h = {h:<6}: 기울기 ≈ {slope:.4f}")

MSE의 두 편미분 $\partial L/\partial w = \frac{1}{n}\sum -2x_i(y_i - \hat{y}_i)$, $\partial L/\partial b = \frac{1}{n}\sum -2(y_i - \hat{y}_i)$를 계산하는 함수입니다. 바닥 $(1.25, 4)$에서는 둘 다 0입니다.

In [ ]:
def gradient(w, b, x, y):
    r = y - (w * x + b)            # 잔차(실제 − 예측)
    gw = np.mean(-2 * x * r)       # MSE를 w로 편미분한 값
    gb = np.mean(-2 * r)           # MSE를 b로 편미분한 값
    return gw, gb

for w, b in [(0.75, 4), (1.25, 2), (1.25, 4)]:
    gw, gb = gradient(w, b, x, y)
    print(f"w = {w}, b = {b}: w 방향 {gw:7.3f}, b 방향 {gb:7.3f}")

곡선과 세 점에서의 접선을 그려 봅니다(이북 8.2절의 접선 그림).

In [ ]:
import matplotlib.pyplot as plt

ws = np.linspace(0.2, 2.3, 200)
plt.plot(ws, [loss(w) for w in ws])
for w0 in [0.75, 1.25, 1.75]:                 # 세 점에서의 접선
    g = gradient(w0, 4, x, y)[0]
    t = np.array([w0 - 0.25, w0 + 0.25])
    plt.plot(t, loss(w0) + g * (t - w0), "--")
plt.xlabel("w"); plt.ylabel("MSE (b = 4)"); plt.ylim(-1, 14)
plt.show()

## 2. 업데이트 규칙과 학습률
$w \leftarrow w - \eta \, \partial L/\partial w$로 다섯 걸음 걸어 봅니다($b$는 4로 고정).

In [ ]:
w, lr = 0.5, 0.02                 # 출발점과 학습률
for step in range(1, 6):
    gw = gradient(w, 4, x, y)[0]  # b는 4로 고정, w 방향 기울기만
    w = w - lr * gw
    print(f"{step}번째: 기울기 {gw:8.4f} → w = {w:.4f}")

학습률을 바꾸어 20번씩 걸어 봅니다. 0.06에서는 발산합니다.

In [ ]:
for lr in [0.002, 0.02, 0.045, 0.06]:
    w = 0.5
    for step in range(20):
        w = w - lr * gradient(w, 4, x, y)[0]
    print(f"학습률 {lr:<5}: 20번 뒤 w = {w:9.4f}")

## 3. 넘파이로 구현하기: 택시 요금
1장의 택시 기록 50건에서 $w$와 $b$를 함께 찾습니다. 목표는 7장의 최소제곱 답 $w = 986.82$, $b = 4{,}953.20$입니다.

In [ ]:
rng = np.random.default_rng(seed=7)        # 1장과 같은 기록 50건
km = rng.uniform(1, 20, size=50).round(1)
fare = (1000 * km + 4800 + rng.normal(0, 600, size=50)).round(-1)

def mse(w, b, x, y):
    return np.mean((y - (w * x + b)) ** 2)

w, b, lr = 0.0, 0.0, 0.005
for step in range(1, 5001):
    gw, gb = gradient(w, b, km, fare)
    w, b = w - lr * gw, b - lr * gb       # 두 값을 함께 바꾼다
    if step in [1, 10, 100, 1000, 5000]:
        print(f"{step:5d}번: w = {w:7.2f}, b = {b:7.2f}, "
              f"MSE {mse(w, b, km, fare):10,.0f}")

이북 8.4절: 학습률을 0.008로 올리면 30번 만에 $w$가 −675,038까지 튕겨 나가며 발산합니다. 직접 확인해 봅니다.

In [ ]:
w, b = 0.0, 0.0
for step in range(1, 31):
    gw, gb = gradient(w, b, km, fare)
    w, b = w - 0.008 * gw, b - 0.008 * gb
    if step % 10 == 0:
        print(f"{step}번: w = {w:,.0f}, b = {b:,.0f}")

거리를 표준화하고 학습률 0.1로 100번만 업데이트합니다. 마지막 줄은 원래 단위로 되돌린 값입니다.

In [ ]:
m, s = km.mean(), km.std()        # 거리의 평균과 표준편차
z = (km - m) / s                  # 표준화한 거리(3장)
print(f"평균 {m:.3f} km, 표준편차 {s:.3f} km")
w, b, lr = 0.0, 0.0, 0.1
for step in range(1, 101):
    gw, gb = gradient(w, b, z, fare)
    w, b = w - lr * gw, b - lr * gb
    if step in [1, 10, 30, 100]:
        print(f"{step:3d}번: w = {w:7.2f}, b = {b:8.2f}, "
              f"MSE {mse(w, b, z, fare):10,.0f}")
print(f"원래 단위로: w = {w / s:.2f}, b = {b - w * m / s:.2f}")

업데이트 횟수에 따라 MSE가 바닥에 다가가는 모습을 비교합니다(두 축 모두 로그 눈금, 이북 8.4절의 그림).

In [ ]:
def run(x, lr, steps):                         # 업데이트마다 MSE를 기록한다
    w, b, hist = 0.0, 0.0, []
    for _ in range(steps):
        gw, gb = gradient(w, b, x, fare)
        w, b = w - lr * gw, b - lr * gb
        hist.append(mse(w, b, x, fare))
    return np.array(hist)

best = mse(986.8243326127581, 4953.2005894925405, km, fare)   # 바닥(최소제곱 답)의 MSE
plt.loglog(run(km, 0.005, 6000) - best, label="raw km, lr 0.005")
plt.loglog(run(z, 0.1, 6000) - best, label="standardized, lr 0.1")
plt.ylim(1e-2, 1e9); plt.xlabel("updates"); plt.ylabel("MSE - best"); plt.legend()
plt.show()

## 4. 배치, 미니배치, 확률적 경사 하강법
배치 크기만 바꾸어 세 방법을 10에포크씩 학습합니다.

In [ ]:
def train(batch_size, lr, epochs, seed=0):
    rng = np.random.default_rng(seed)
    w, b, n = 0.0, 0.0, len(z)
    for epoch in range(epochs):
        idx = rng.permutation(n)            # 순서를 섞는다
        for start in range(0, n, batch_size):
            j = idx[start:start + batch_size]   # 미니배치
            gw, gb = gradient(w, b, z[j], fare[j])
            w, b = w - lr * gw, b - lr * gb
    return w / s, b - w * m / s             # 원래 단위로

for name, size, lr in [("배치", 50, 0.1), ("미니배치", 10, 0.1),
                       ("확률적", 1, 0.01)]:
    w, b = train(size, lr, epochs=10)
    print(f"{name}(크기 {size}): 업데이트 {10 * 50 // size}번, "
          f"w = {w:.2f}, b = {b:.2f}, "
          f"MSE {mse(w, b, km, fare):,.0f}")

사이킷런의 `SGDRegressor`를 표준화 없이, 그리고 표준화하고 학습합니다. `penalty=None`은 규제(13장)를 끄는 설정입니다.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = km.reshape(-1, 1)
raw = SGDRegressor(penalty=None, random_state=0).fit(X, fare)
print(f"표준화 없이: w = {raw.coef_[0]:.2f}, "
      f"b = {raw.intercept_[0]:.2f}")
sgd = make_pipeline(StandardScaler(),
                    SGDRegressor(penalty=None, random_state=0))
sgd.fit(X, fare)
sc, reg = sgd[0], sgd[1]                  # 스케일러와 모델
w = reg.coef_[0] / sc.scale_[0]           # 원래 단위로
b = reg.intercept_[0] - w * sc.mean_[0]
print(f"표준화하고:  w = {w:.2f}, b = {b:.2f} "
      f"({reg.n_iter_}에포크)")

---
## 실습 문제

**문제 1.** 표준화한 택시 기록(`z`, `fare`)에서 학습률을 0.01, 0.1, 0.5, 0.9, 1.0, 1.1로 바꾸어 보세요. 학습률마다 MSE가 바닥(`best`)과 1 이내로 가까워질 때까지 몇 번 업데이트해야 하는지(최대 10,000번) 세고, 발산하는 학습률을 찾으세요. 학습률 0.5에서는 왜 그렇게 빨리 끝날까요?

In [ ]:
# 문제 1 풀이

**문제 2.** 펭귄의 날개 길이, 부리 길이, 부리 깊이 세 특징으로 몸무게를 예측하는 다중 선형 회귀를 경사 하강법으로 학습시키세요. 세 특징은 표준화하고, 가중치 $w_j$의 기울기는 $\frac{1}{n}\sum -2x_{ij}(y_i - \hat{y}_i)$로 계산합니다. 학습률 0.1로 1,000번 업데이트한 가중치와 절편을, 같은 표준화 데이터로 학습한 `LinearRegression`의 계수와 비교하세요. (펭귄 342마리 전체를 씁니다.)

In [ ]:
# 문제 2 풀이

**문제 3.** 표준화한 택시 기록에서 배치 크기를 1, 5, 10, 25, 50으로 바꾸어 학습률 0.05로 20에포크씩 학습하세요(`train` 함수를 쓰세요). 배치 크기마다 `seed`를 0\~4로 바꾸어 다섯 번 학습하고, 원래 단위의 $w$가 얼마나 흩어지는지(표준편차) 비교하세요.

In [ ]:
# 문제 3 풀이

**문제 4.** 손실을 MSE 대신 MAE로 바꾸어 경사 하강법을 해 보세요. 기록 하나의 $|y - \hat{y}|$를 $w$로 미분하면 $-x \cdot \mathrm{sign}(y - \hat{y})$이고(`np.sign`), $b$로 미분하면 $-\mathrm{sign}(y - \hat{y})$입니다. 표준화한 택시 기록에서 학습률을 ① 100으로 고정, ② $1000/\sqrt{t}$($t$는 업데이트 횟수)로 줄여 가며 각각 3,000번 업데이트하고, 마지막 10번의 $w$(표준화한 단위)가 얼마나 흔들리는지 비교하세요.

In [ ]:
# 문제 4 풀이